# Lab 09-10 · Short-Channel Effects and Leakage

Companion to chapter [09-10](../../chapters/09-10-short-channel-effects-and-leakage.md). Handout: [README.md](README.md).

The transistor of Parts 1–2: 2 nm EOT, $m$ = 1.2, $V_t$ = 0.35 V, $\mu_{ns}$ = 183 cm²/V·s, $v_{sat}$ = 8 × 10⁶ cm/s, $V_{GS}$ = 1.0 V, $W$ = 1 µm, at 300 K.

In [1]:
import math
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q = 1.602176634e-19
VT = 1.380649e-23 * 300 / Q
EPS = 11.7 * 8.8541878128e-14
EOX = 3.9 * 8.8541878128e-14
COX = EOX / 2e-7
M, VTH, MU, VSAT, W = 1.2, 0.35, 183.0, 8e6, 1e-4
VOV = 1.0 - VTH
ESAT = 2 * VSAT / MU
print(f"C_ox = {COX*1e6:.4f} uF/cm^2, E_sat = {ESAT:.4g} V/cm")

C_ox = 1.7266 uF/cm^2, E_sat = 8.743e+04 V/cm


## Part 1 — Velocity saturation and $I_{Dsat}$ against length

Below saturation, $I_D(L + V_{DS}/\mathcal E_{sat}) = WC_{ox}\mu_{ns}\int_0^{V_{DS}}(V_{ov} - mV)\,dV$. The local field follows from $I_D(1 + \mathcal E/\mathcal E_{sat}) = WC_{ox}\mu_{ns}(V_{ov} - mV_c)\mathcal E$. The current saturates when the field at the drain end reaches $\mathcal E_{sat}$ (where $v = v_{sat}$).

*Predict first:* from 2 µm to 20 nm, by what factor does $I_{Dsat}$ rise (the long-channel law says 100)?

In [2]:
def i_lin(vds, L, vov=VOV):
    integral = quad(lambda V: vov - M * V, 0, vds, epsabs=0, epsrel=1e-12)[0]
    return W * COX * MU * integral / (L + vds / ESAT)

def drain_field(vds, L, vov=VOV):
    I = i_lin(vds, L, vov)
    return I / (W * COX * MU * (vov - M * vds) - I / ESAT)

def idsat(L, vov=VOV):
    # the drain-end field diverges where W C mu (vov - m V) = I/E_sat; bracket below that pole
    pole = brentq(lambda v: W * COX * MU * (vov - M * v) - i_lin(v, L, vov) / ESAT, 0, vov / M)
    vd = brentq(lambda v: drain_field(v, L, vov) - ESAT, 1e-9, pole * (1 - 1e-12))
    return i_lin(vd, L, vov), vd

I50, vd50 = idsat(50e-7)
print(f"L = 50 nm: V_Dsat = {vd50:.4f} V, I_Dsat = {I50*1e3:.4f} mA/um")
assert abs(I50 * 1e3 - 0.497) < 5e-4

long = lambda L, vov=VOV: W / (2 * M * L) * MU * COX * vov ** 2
I2u, I20 = idsat(2e-4)[0], idsat(20e-7)[0]
rise = I20 / I2u
print(f"I_Dsat: 2 um {I2u*1e3:.4f}, 20 nm {I20*1e3:.4f} mA/um; rise {rise:.2f} (long-channel law: {long(20e-7)/long(2e-4):.0f})")
assert abs(I2u * 1e3 - 0.0270) < 5e-5 and abs(I20 * 1e3 - 0.679) < 5e-4 and abs(rise - 25.2) < 0.05
Lhalf = brentq(lambda L: idsat(L)[0] / long(L) - 0.5, 10e-7, 1e-4)
print(f"velocity saturation halves the long-channel current at L = {Lhalf*1e7:.1f} nm")
assert abs(Lhalf * 1e7 - 62.0) < 0.05

Ls = np.logspace(np.log10(20e-7), np.log10(2e-4), 40)
plt.loglog(Ls * 1e7, [long(L) * 1e3 for L in Ls], "--", label="long-channel")
plt.loglog(Ls * 1e7, [idsat(L)[0] * 1e3 for L in Ls], label="velocity saturation")
plt.loglog(Ls * 1e7, [W * 1.6e7 * COX * VOV * 1e3] * len(Ls), ":", label="injection limit, B = 1")
plt.xlabel("L (nm)"); plt.ylabel("I_Dsat (mA/um)"); plt.legend(); plt.show()

L = 50 nm: V_Dsat = 0.2419 V, I_Dsat = 0.4968 mA/um
I_Dsat: 2 um 0.0270, 20 nm 0.6787 mA/um; rise 25.16 (long-channel law: 100)
velocity saturation halves the long-channel current at L = 62.0 nm


## Part 2 — Series resistance

With a source resistance $R_S$, the internal overdrive is $V_{ov} - I_DR_S$. Solve $I = I_{Dsat}(V_{ov} - IR_S)$.

*Predict first:* does 100 Ω·µm cost more or less than 10% of the current at 50 nm?

In [3]:
RS = 100.0
I0 = I50
Isc = brentq(lambda I: idsat(50e-7, VOV - I * RS)[0] - I, 1e-6, I0)
print(f"self-consistent I_Dsat = {Isc*1e3:.4f} mA/um, loss {100*(1-Isc/I0):.1f}%; Hu closed form {I0/(1+RS*I0/VOV)*1e3:.4f}")
assert abs(Isc * 1e3 - 0.448) < 5e-4
vds = np.linspace(0, 1.0, 101)
for vgs in (0.6, 0.8, 1.0):
    vov = vgs - VTH
    Is, vd = idsat(50e-7, vov)
    plt.plot(vds, [i_lin(v, 50e-7, vov) * 1e3 if v < vd else Is * 1e3 for v in vds], label=f"V_GS = {vgs} V")
plt.xlabel("V_DS (V)"); plt.ylabel("I_D (mA/um)"); plt.legend(); plt.show()

self-consistent I_Dsat = 0.4478 mA/um, loss 9.9%; Hu closed form 0.4616


## Part 3 — Threshold roll-off and DIBL

Hu's model: $V_t = V_{t,\text{long}} - (V_{DS} + 0.4)e^{-L/l_d}$, with $l_d$ = 10 nm here. Subthreshold current: $I = I_t10^{(V_{GS} - V_t)/S}$, $S$ = 80 mV/decade.

*Predict first:* at $L$ = 30 nm, by what factor does raising $V_{DS}$ from 0.05 V to 1.0 V raise the off-current?

In [4]:
LD, S = 10e-7, 0.080
dvt = lambda L, vds: (vds + 0.4) * math.exp(-L / LD)
shift = dvt(30e-7, 1.0) - dvt(30e-7, 0.05)
factor = 10 ** (shift / S)
print(f"threshold shift at 30 nm: {shift*1e3:.1f} mV; off-current factor {factor:.2f}; at 100 nm: {(dvt(100e-7,1.0)-dvt(100e-7,0.05))*1e3:.4f} mV")
assert abs(factor - 3.90) < 5e-3 and abs(shift * 1e3 - 47.3) < 0.05
Ls = np.linspace(15e-7, 100e-7, 100)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(Ls * 1e7, [dvt(L, 0.05) * 1e3 for L in Ls], label="V_DS = 0.05 V")
ax[0].plot(Ls * 1e7, [dvt(L, 1.0) * 1e3 for L in Ls], label="V_DS = 1.0 V")
ax[0].set_xlabel("L (nm)"); ax[0].set_ylabel("roll-off (mV)"); ax[0].legend()
vg = np.linspace(0, 0.6, 100)
for vds in (0.05, 1.0):
    vt = 0.35 - dvt(30e-7, vds)
    ax[1].semilogy(vg, 1e-7 * 10 ** ((vg - vt) / S), label=f"V_DS = {vds} V")
ax[1].set_xlabel("V_GS (V)"); ax[1].set_ylabel("I_D (A), I_t = 100 nA"); ax[1].legend(); plt.show()

threshold shift at 30 nm: 47.3 mV; off-current factor 3.90; at 100 nm: 0.0431 mV


## Part 4 — The I_on–I_off window

Keep a fixed overdrive $V_{DD} - V_t$ for speed; the on/off ratio below threshold is then $10^{V_t/S}$.

*Predict first:* with $V_{DD}$ = 0.6 V, 0.3 V of overdrive and $S$ = 70 mV/decade, how many decades?

In [5]:
decades = lambda vdd, ov, s: (vdd - ov) / s
d = decades(0.6, 0.3, 0.070)
print(f"{d:.2f} decades of on/off ratio below threshold")
assert abs(d - 4.29) < 5e-3
for T in (300, 358.15, 398.15):
    s = 0.070 * T / 300
    print(f"T = {T:.0f} K: S = {s*1e3:.1f} mV/dec, decades = {decades(0.6, 0.3, s):.2f}")
vdds = np.linspace(0.4, 1.0, 50)
for s in (0.060, 0.070, 0.090):
    plt.plot(vdds, [decades(v, 0.3, s) for v in vdds], label=f"S = {s*1e3:.0f} mV/dec")
plt.xlabel("V_DD (V)"); plt.ylabel("decades below threshold"); plt.legend(); plt.show()

4.29 decades of on/off ratio below threshold
T = 300 K: S = 70.0 mV/dec, decades = 4.29
T = 358 K: S = 83.6 mV/dec, decades = 3.59
T = 398 K: S = 92.9 mV/dec, decades = 3.23


## Part 5 — Random dopant fluctuation

Example 09-10.6: $W$ = 50 nm, $L$ = 30 nm, $W_{dep}$ = 20 nm, $N_A$ = 5 × 10¹⁸ cm⁻³. Each device gets a Poisson number of dopants at uniform random depths; each shifts $V_t$ by $q(1 - x/W_{dep})/(C_{ox}WL)$.

*Predict first:* will the spread match 75.8 mV or 43.7 mV?

In [6]:
rng = np.random.default_rng(910)
Wg, Lg, Wd, NA = 50e-7, 30e-7, 20e-7, 5e18
nbar = NA * Wg * Lg * Wd
counts = rng.poisson(nbar, 20000)
dv = np.array([Q * np.sum(1 - rng.uniform(0, 1, n)) / (COX * Wg * Lg) for n in counts])
sigma_mc = dv.std() * 1e3
sigma_w = Q / COX * math.sqrt(NA * Wd / (3 * Wg * Lg)) * 1e3
print(f"mean count {nbar:.0f}; Monte Carlo sigma = {sigma_mc:.1f} mV; depth-weighted formula {sigma_w:.1f} mV")
assert abs(sigma_w - 43.7) < 0.05 and abs(sigma_mc / sigma_w - 1) < 0.02
plt.hist(dv * 1e3 - dv.mean() * 1e3, bins=60)
plt.xlabel("threshold deviation (mV)"); plt.ylabel("devices"); plt.show()

mean count 150; Monte Carlo sigma = 43.6 mV; depth-weighted formula 43.7 mV
